# Notebook gỡ lỗi pipeline WSSS trên BTXRD — cấu hình `btxrd_best`

Notebook này được thiết kế để **quan sát và gỡ lỗi từng bước** của pipeline tạo pseudo-mask từ ảnh X-quang BTXRD.

Khác với một notebook huấn luyện rút gọn, phiên bản này ưu tiên:

- Chia mỗi thao tác thành các cell nhỏ, dễ chạy lại độc lập.
- Giải thích rõ mục đích, đầu vào và đầu ra của từng cell.
- Hiển thị ảnh trung gian ở các bước quan trọng.
- Phân tách rõ hai giao thức `predicted` và `ground_truth`.
- Không viết lại pipeline sản xuất bên trong notebook; notebook gọi trực tiếp các script hiện có trong repository.

> **Nguyên tắc quan trọng:** polygon ground truth chỉ được dùng trong các cell trực quan hóa, đánh giá hoặc baseline oracle có giám sát. Polygon không được truyền vào classifier, CAM, prompt SAM, quá trình chọn candidate hoặc hậu xử lý pseudo-mask.


## Tổng quan pipeline

```text
Ảnh X-quang BTXRD
    ↓
DenseNet121 phân loại tumor_type ở mức ảnh
    ↓
LayerCAM từ denseblock2 / denseblock3 / denseblock4
    ↓
CAM tương phản giữa lớp khối u và lớp normal
    ↓
Ngưỡng percentile 85 / 90 / 95
    ↓
Tách tối đa 3 connected components
    ↓
Sinh bounding box + positive points + negative points
    ↓
SAM ViT-B ở kích thước 512 px
    ↓
Tập candidate từ box, point và box+point
    ↓
Chấm điểm, chọn candidate tốt nhất
    ↓
Hậu xử lý morphology
    ↓
Pseudo tumor mask
```

Notebook sử dụng profile chuẩn `btxrd_best` để giữ cấu hình nhất quán với code hiện tại.


## Thứ tự chạy đề xuất

1. Khai báo đường dẫn và các công tắc chạy.
2. Chuẩn bị repository, dependency và kiểm tra GPU.
3. Tìm dataset BTXRD, kiểm tra split và rò rỉ nhãn.
4. Trực quan hóa polygon ground truth cho mục đích chẩn đoán.
5. Chuẩn bị checkpoint SAM và huấn luyện classifier.
6. Kiểm tra log huấn luyện và snapshot CAM.
7. Truy vết một ảnh từ classifier đến prompt trước SAM.
8. Chạy generator thật ở chế độ `--debug` để xem candidate SAM.
9. Chạy toàn bộ validation với giao thức `predicted`.
10. Chạy giao thức `ground_truth` riêng để chẩn đoán khả năng định vị.
11. Tổng hợp metric, phân rã lỗi và lưu manifest tái lập.

Các bước tốn thời gian được điều khiển bằng công tắc ở phần đầu notebook. Không sử dụng tập test để tinh chỉnh tham số.


# 0. Cấu hình chung

Phần này chỉ khai báo thư viện chuẩn, đường dẫn và công tắc chạy. Việc tách riêng giúp bạn thay đổi môi trường hoặc bật/tắt một giai đoạn mà không phải sửa các cell xử lý phía sau.


### Cell 0.1 — Import thư viện chuẩn

**Mục đích:** nạp các thư viện dùng để thao tác đường dẫn, JSON, biến môi trường, chạy lệnh shell và quản lý Python path.

**Đầu ra:** các module chuẩn sẵn sàng cho toàn bộ notebook.


In [ ]:
# Thư viện chuẩn dùng xuyên suốt notebook
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys


### Cell 0.2 — Xác định thư mục làm việc và repository

**Mục đích:** tự động thích nghi giữa Kaggle và môi trường local.

- Nếu thư mục hiện tại đã chứa `project/`, notebook dùng code tại chỗ.
- Nếu chưa có, notebook dự kiến clone repository vào `/kaggle/working/Thesis`.
- `PROJECT_DIR` luôn trỏ tới thư mục chứa các script như `train_classifier.py`.

**Có thể ghi đè bằng biến môi trường:**

- `BTXRD_REPO_URL`
- `BTXRD_GIT_BRANCH`
- `BTXRD_ROOT`


In [ ]:
# Thư mục notebook hiện tại
NOTEBOOK_ROOT = Path.cwd()

# Các thư mục chuẩn trên Kaggle
KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

# Thư mục mặc định để lưu kết quả
DEFAULT_WORKING = (
    KAGGLE_WORKING
    if KAGGLE_WORKING.exists()
    else NOTEBOOK_ROOT / "notebook_runs"
)

# Thông tin repository có thể ghi đè qua biến môi trường
REPO_URL = os.environ.get(
    "BTXRD_REPO_URL",
    "https://github.com/itsthang333/Thesis.git",
)
GIT_BRANCH = os.environ.get("BTXRD_GIT_BRANCH", "TN_exp")
DATASET_OVERRIDE = os.environ.get("BTXRD_ROOT", "")

# Xác định vị trí repository
if (NOTEBOOK_ROOT / "project").exists():
    PROJECT_PARENT = NOTEBOOK_ROOT
else:
    PROJECT_PARENT = KAGGLE_WORKING / "Thesis"

PROJECT_DIR = PROJECT_PARENT / "project"


### Cell 0.3 — Khai báo thư mục đầu ra

**Mục đích:** tách kết quả theo từng giao thức để tránh ghi đè hoặc trộn lẫn.

- `classifier_btxrd_best`: checkpoint và log classifier.
- `pseudo_predicted`: pseudo-mask dùng lớp do classifier dự đoán.
- `pseudo_ground_truth`: kết quả định vị dùng nhãn ảnh-level thật.
- `single_image_debug`: artifact chi tiết cho một ảnh.
- `evaluations`: CSV đánh giá.


In [ ]:
# Gốc lưu toàn bộ kết quả của notebook
OUTPUT_ROOT = Path(
    os.environ.get(
        "BTXRD_OUTPUT",
        str(DEFAULT_WORKING / "btxrd_debug"),
    )
)

# Các thư mục con được tách theo chức năng
CLASSIFIER_OUTPUT = OUTPUT_ROOT / "classifier_btxrd_best"
PREDICTED_OUTPUT = OUTPUT_ROOT / "pseudo_predicted"
GROUND_TRUTH_UNET_OUTPUT = OUTPUT_ROOT / "unet_trained_on_ground_truth"
UNET_EVAL_OUTPUT = OUTPUT_ROOT / "unet_evaluations"
GROUND_TRUTH_OUTPUT = OUTPUT_ROOT / "pseudo_ground_truth"
DEBUG_OUTPUT = OUTPUT_ROOT / "single_image_debug"
EVAL_OUTPUT = OUTPUT_ROOT / "evaluations"

### Cell 0.4 — Khai báo checkpoint SAM và kích thước ảnh

**Mục đích:** gom các hằng số quan trọng vào một nơi để dễ kiểm tra.

- Classifier nhận ảnh `320 × 320`.
- SAM nhận prompt trên ảnh đã đưa về kích thước `512`.
- `SAM_CHECKPOINT` có thể được chỉ định bằng biến môi trường cùng tên.


In [ ]:
# Ưu tiên checkpoint SAM đặt cạnh notebook; nếu chưa có sẽ dùng thư mục output
DEFAULT_SAM = (
    NOTEBOOK_ROOT / "sam_vit_b_01ec64.pth"
    if (NOTEBOOK_ROOT / "sam_vit_b_01ec64.pth").exists()
    else OUTPUT_ROOT / "sam_vit_b_01ec64.pth"
)

SAM_CHECKPOINT = Path(
    os.environ.get("SAM_CHECKPOINT", str(DEFAULT_SAM))
)

# Kích thước ảnh và số worker
IMAGE_SIZE = 320
SAM_IMAGE_SIZE = 512
NUM_WORKERS = int(os.environ.get("BTXRD_NUM_WORKERS", "2"))

# Có thể chỉ định trước tên ảnh cần debug; để trống sẽ tự chọn từ validation
DEBUG_IMAGE_NAME = os.environ.get("BTXRD_DEBUG_IMAGE", "")


### Cell 0.5 — Công tắc điều khiển các giai đoạn

**Mục đích:** cho phép chuẩn bị lệnh nhưng không nhất thiết chạy ngay các bước tốn thời gian.

Đặt một biến thành `False` khi bạn chỉ muốn kiểm tra cell, xem lệnh hoặc sử dụng artifact đã có từ lần chạy trước.


In [ ]:
# Cài dependency từ requirements.txt
INSTALL_DEPENDENCIES = True

# Các công tắc nguyên bản của pipeline pseudo-mask/WSSS
RUN_TRAIN_CLASSIFIER = True
RUN_SINGLE_IMAGE_DEBUG = True
RUN_FULL_PREDICTED = True
RUN_FULL_GROUND_TRUTH = True

# Nhánh fully supervised độc lập, chỉ dùng để đối chứng
RUN_TRAIN_GROUND_TRUTH_UNET = True
RUN_EVALUATE_GROUND_TRUTH_UNET = True

# Dành cho báo cáo test cuối cùng; notebook debug không tự bật
RUN_TEST_REPORT = False

### Cell 0.6 — Tạo thư mục đầu ra và in cấu hình

**Mục đích:** tạo trước toàn bộ cây thư mục cần dùng và hiển thị các đường dẫn quan trọng.

**Kiểm tra mong đợi:** `PROJECT_DIR`, branch và `OUTPUT_ROOT` phải đúng với môi trường đang chạy.


In [ ]:
# Tạo tất cả thư mục đầu ra nếu chưa tồn tại
for path in [
    OUTPUT_ROOT,
    CLASSIFIER_OUTPUT,
    PREDICTED_OUTPUT,
    GROUND_TRUTH_UNET_OUTPUT,
    UNET_EVAL_OUTPUT,
    GROUND_TRUTH_OUTPUT,
    DEBUG_OUTPUT,
    EVAL_OUTPUT,
]:
    path.mkdir(parents=True, exist_ok=True)

print("Thư mục project:", PROJECT_DIR)
print("Git branch:", GIT_BRANCH)
print("Thư mục lưu kết quả:", OUTPUT_ROOT)

# 1. Chuẩn bị repository và môi trường

Mục tiêu của phần này là đảm bảo notebook gọi đúng code trong repository hiện tại thay vì chứa một bản sao pipeline riêng biệt.


### Cell 1.1 — Clone repository khi cần

**Mục đích:** chỉ clone repository khi `PROJECT_DIR` chưa tồn tại.

**Lưu ý:** cell này không tự động `git pull` nếu repository đã có, nhằm tránh thay đổi code ngoài ý muốn trong lúc debug.


In [ ]:
# Clone đúng branch khi repository chưa có trong môi trường hiện tại
if not PROJECT_DIR.exists():
    PROJECT_PARENT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            GIT_BRANCH,
            REPO_URL,
            str(PROJECT_PARENT),
        ],
        check=True,
    )

print("Repository đã sẵn sàng tại:", PROJECT_PARENT)


### Cell 1.2 — Chuyển thư mục làm việc vào `project/`

**Mục đích:** đảm bảo các import nội bộ và script CLI sử dụng đúng đường dẫn tương đối.

**Đầu ra:** `Path.cwd()` phải trùng với `PROJECT_DIR`.


In [ ]:
# Chuyển vào thư mục chứa source code
os.chdir(PROJECT_DIR)

# Cho phép import trực tiếp các module trong project
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("Thư mục làm việc hiện tại:", Path.cwd())


### Cell 1.3 — Hàm chạy lệnh và truyền log theo thời gian thực

**Mục đích:** chạy script Python bên ngoài notebook nhưng vẫn hiển thị log liên tục.

Hàm tự thêm cờ `-u` cho Python để stdout không bị buffer. Nếu lệnh trả mã lỗi khác 0, hàm ném `CalledProcessError`.


In [ ]:
def run_streaming(cmd, cwd=PROJECT_DIR, check=True):
    """Chạy lệnh và in stdout/stderr theo thời gian thực."""
    cmd = [str(item) for item in cmd]

    # Bật chế độ unbuffered cho lệnh Python
    if cmd and Path(cmd[0]).name.lower().startswith("python"):
        cmd = [cmd[0], "-u", *cmd[1:]]

    print("$", " ".join(shlex.quote(item) for item in cmd))

    process = subprocess.Popen(
        cmd,
        cwd=str(cwd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="")

    return_code = process.wait()

    if check and return_code:
        raise subprocess.CalledProcessError(return_code, cmd)

    return return_code


### Cell 1.4 — Cài dependency

**Mục đích:** cài package từ `requirements.txt` của repository và bổ sung các package dùng để phân tích trong notebook.

Đặt `INSTALL_DEPENDENCIES=False` nếu môi trường đã cài đủ để tránh mất thời gian cài lại.


In [ ]:
if INSTALL_DEPENDENCIES:
    run_streaming(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-r",
            str(PROJECT_DIR / "requirements.txt"),
        ]
    )

    run_streaming(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "pandas",
            "openpyxl",
            "opencv-python",
        ]
    )
else:
    print("Bỏ qua cài dependency vì INSTALL_DEPENDENCIES=False.")


### Cell 1.5 — Import thư viện phân tích và deep learning

**Mục đích:** nạp NumPy, Pandas, PIL, Matplotlib và PyTorch sau khi dependency đã sẵn sàng.


In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import torch


### Cell 1.6 — Kiểm tra phiên bản Python, PyTorch và GPU

**Mục đích:** xác nhận runtime có nhận CUDA hay không và xem dung lượng VRAM của từng GPU.

Khi `CUDA available=False`, các bước huấn luyện hoặc chạy SAM có thể rất chậm.


In [ ]:
print("Phiên bản Python:", sys.version)
print("Phiên bản PyTorch:", torch.__version__)
print("CUDA khả dụng:", torch.cuda.is_available())
print("Số thiết bị CUDA:", torch.cuda.device_count())

if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        properties = torch.cuda.get_device_properties(index)
        memory_gib = round(properties.total_memory / 2**30, 2)

        print(
            f"GPU {index}:",
            torch.cuda.get_device_name(index),
            f"- {memory_gib} GiB",
        )


# 2. Tìm dataset, tạo split và kiểm tra rò rỉ nhãn

BTXRD không có split cố định trong pipeline này. Loader tạo split `80/10/10`, phân tầng theo toàn bộ 10 lớp `tumor_type`, với seed 42.


### Cell 2.1 — Import loader BTXRD

**Mục đích:** nạp các hàm chính thức từ code hiện tại.

- `load_btxrd_records`: đọc metadata.
- `resolve_btxrd_root`: xác nhận cấu trúc thư mục dataset.
- `split_btxrd_records`: tạo split xác định.
- `build_classification_dataset`: dataset chỉ dùng nhãn mức ảnh.
- `build_segmentation_dataset`: dataset polygon, chỉ dùng cho chẩn đoán/đánh giá.


In [ ]:
from datasets.btxrd import (
    TUMOR_TYPE_CLASS_NAMES,
    BTXRDSegmentationDataset,
    load_btxrd_records,
    resolve_btxrd_root,
    split_btxrd_records,
)
from datasets.factory import (
    build_classification_dataset,
    build_segmentation_dataset,
)


### Cell 2.2 — Hàm tìm thư mục gốc BTXRD

**Mục đích:** tìm dataset trong thư mục chỉ định và tối đa hai cấp con.

Mỗi candidate được kiểm tra bằng `resolve_btxrd_root`, do đó chỉ thư mục có cấu trúc BTXRD hợp lệ mới được chấp nhận.


In [ ]:
def find_btxrd_root(base: Path):
    """Tìm thư mục BTXRD hợp lệ trong base và tối đa hai cấp con."""
    if not base or not base.exists():
        return None

    candidates = [
        base,
        *base.glob("*"),
        *base.glob("*/*"),
    ]

    for candidate in candidates:
        try:
            return resolve_btxrd_root(candidate)
        except FileNotFoundError:
            continue

    return None


### Cell 2.3 — Xác định vị trí dataset

**Mục đích:** ưu tiên đường dẫn trong `BTXRD_ROOT`; nếu không có thì tìm trong `/kaggle/input` hoặc thư mục notebook.

**Lỗi thường gặp:** dataset chưa được attach vào Kaggle hoặc cấu trúc thiếu `images/`, `Annotations/`, `dataset.csv`/`dataset.xlsx`.


In [ ]:
# Ưu tiên đường dẫn do người dùng chỉ định
if DATASET_OVERRIDE:
    search_root = Path(DATASET_OVERRIDE)
elif KAGGLE_INPUT.exists():
    search_root = KAGGLE_INPUT
else:
    search_root = NOTEBOOK_ROOT

BTXRD_ROOT = find_btxrd_root(search_root)

if BTXRD_ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy BTXRD. Hãy đặt BTXRD_ROOT hoặc attach "
        "dataset có images/, Annotations/ và dataset.csv/xlsx."
    )

print("Thư mục BTXRD:", BTXRD_ROOT)


### Cell 2.4 — Đọc toàn bộ record

**Mục đích:** tải metadata đã được chuẩn hóa bởi loader hiện tại.

**Đầu ra:** `records` là danh sách record dùng chung để tạo train/validation/test.


In [ ]:
records = load_btxrd_records(BTXRD_ROOT)

print("Tổng số record:", len(records))
print("Ví dụ record đầu tiên:")
display(pd.Series(records[0]).to_frame("giá trị"))


### Cell 2.5 — Hàm chuyển một split thành DataFrame

**Mục đích:** tạo split bằng seed 42 và bổ sung tên lớp dễ đọc từ chỉ số `tumor_type`.


In [ ]:
def split_frame(split_name):
    """Tạo DataFrame cho một split BTXRD xác định bằng seed 42."""
    split_rows = split_btxrd_records(
        records,
        split=split_name,
        seed=42,
    )

    frame = pd.DataFrame(split_rows)

    class_mapping = dict(enumerate(TUMOR_TYPE_CLASS_NAMES))
    frame["tumor_type_name"] = frame["tumor_type"].map(class_mapping)

    return frame


### Cell 2.6 — Tạo ba split

**Mục đích:** lưu DataFrame của `train`, `val` và `test` trong một dictionary để các cell sau truy cập nhất quán.


In [ ]:
split_frames = {
    split_name: split_frame(split_name)
    for split_name in ["train", "val", "test"]
}

for split_name, frame in split_frames.items():
    print(f"{split_name}: {len(frame)} ảnh")


### Cell 2.7 — Tổng hợp số lượng lớp theo split

**Mục đích:** kiểm tra phân tầng có giữ được phân bố của cả 10 lớp hay không.

**Đầu ra:** bảng `split_summary` ở dạng dài và bảng pivot theo lớp.


In [ ]:
summary_parts = []

for split_name, frame in split_frames.items():
    class_counts = (
        frame["tumor_type_name"]
        .value_counts()
        .reindex(TUMOR_TYPE_CLASS_NAMES, fill_value=0)
    )

    summary_parts.append(
        pd.DataFrame(
            {
                "split": split_name,
                "class": class_counts.index,
                "count": class_counts.values,
            }
        )
    )

    tumor_count = int((frame["tumor_type"] > 0).sum())
    normal_count = int((frame["tumor_type"] == 0).sum())

    print(
        f"{split_name}: n={len(frame)}, "
        f"tumor={tumor_count}, normal={normal_count}"
    )

split_summary = pd.concat(summary_parts, ignore_index=True)
split_pivot = split_summary.pivot(
    index="class",
    columns="split",
    values="count",
)

display(split_pivot)


### Cell 2.8 — Vẽ biểu đồ phân bố lớp

**Mục đích:** quan sát trực quan độ cân bằng giữa các split và phát hiện lớp có quá ít mẫu.


In [ ]:
split_pivot.plot.bar(
    figsize=(14, 4),
    grid=True,
)

plt.title("Phân tầng BTXRD 80/10/10 theo tumor_type")
plt.xlabel("Lớp tumor_type")
plt.ylabel("Số ảnh")
plt.tight_layout()
plt.show()


### Cell 2.9 — Chọn ảnh validation để debug

**Mục đích:** đảm bảo các bước truy vết sử dụng cùng một ảnh xác định.

- Nếu `BTXRD_DEBUG_IMAGE` đã được đặt, notebook dùng ảnh đó.
- Nếu chưa, notebook chọn ảnh tumor đầu tiên trong validation.
- Tên ảnh được ghi vào file để truyền cho script `generate_pseudo_masks.py`.


In [ ]:
if not DEBUG_IMAGE_NAME:
    DEBUG_IMAGE_NAME = str(
        split_frames["val"]
        .query("tumor_type > 0")
        .iloc[0]["image_id"]
    )

DEBUG_IMAGE_LIST = OUTPUT_ROOT / "debug_image_list.txt"
DEBUG_IMAGE_LIST.write_text(
    DEBUG_IMAGE_NAME + "\n",
    encoding="utf-8",
)

print("Ảnh được chọn để debug:", DEBUG_IMAGE_NAME)
print("File danh sách ảnh:", DEBUG_IMAGE_LIST)


### Cell 2.10 — Khởi tạo dataset classification validation

**Mục đích:** tạo dataset chỉ sử dụng `tumor_type` ở mức ảnh. Đây là loại dataset hợp lệ cho classifier và quá trình sinh CAM.


In [ ]:
classification_ds = build_classification_dataset(
    "btxrd",
    root=BTXRD_ROOT,
    split="val",
    target_columns=["tumor_type"],
    image_size=IMAGE_SIZE,
)

print("Số mẫu classification validation:", len(classification_ds))
print("Cột nhãn:", classification_ds.target_columns)


### Cell 2.11 — Khởi tạo dataset segmentation validation

**Mục đích:** đọc polygon mask phục vụ trực quan hóa và đánh giá.

> Dataset này không được truyền vào generator pseudo-mask.


In [ ]:
segmentation_ds = build_segmentation_dataset(
    "btxrd",
    root=BTXRD_ROOT,
    split="val",
    image_size=IMAGE_SIZE,
    augment=False,
)

print("Số mẫu segmentation validation:", len(segmentation_ds))


### Cell 2.12 — Kiểm tra ID và rào chắn chống rò rỉ

**Mục đích:** xác nhận hai dataset tham chiếu cùng tập ảnh nhưng classification dataset chỉ chứa nhãn `tumor_type`.

Assertion giúp notebook dừng sớm nếu cấu hình nhãn classification bị thay đổi ngoài ý muốn.


In [ ]:
classification_names = {
    str(sample["image_id"])
    for sample in classification_ds.samples
}
segmentation_names = {
    str(sample["image_id"])
    for sample in segmentation_ds.samples
}

print("Số ID classification:", len(classification_names))
print("Số ID segmentation:", len(segmentation_names))
print("Hai tập ID giống nhau:", classification_names == segmentation_names)

assert classification_ds.target_columns == ["tumor_type"]

print(
    "Generator chỉ dùng nhãn mức ảnh:",
    classification_ds.target_columns,
)
print(
    "Polygon mask chỉ tồn tại trong các cell chẩn đoán/đánh giá."
)


# 3. Trực quan hóa polygon ground truth

Phần này giúp kiểm tra chất lượng annotation. Đây là bước chẩn đoán độc lập và không phải một phần của quá trình tạo pseudo-mask.


### Cell 3.1 — Chọn một record tumor hoặc normal

**Mục đích:** lọc validation theo `tumor_type` và lấy một record đại diện.

Cell này chỉ chọn metadata, chưa đọc ảnh hoặc polygon.


In [ ]:
def select_gt_row(want_tumor):
    """Chọn một hàng tumor hoặc normal từ validation."""
    validation_frame = split_frames["val"]

    if want_tumor:
        candidates = validation_frame[validation_frame["tumor_type"] > 0]
    else:
        candidates = validation_frame[validation_frame["tumor_type"] == 0]

    return candidates.iloc[0]


### Cell 3.1a — Nạp ảnh và polygon của hàng đã chọn

**Mục đích:** tách thao tác đọc dữ liệu khỏi thao tác vẽ.


In [ ]:
def load_gt_case(want_tumor):
    """Nạp ảnh RGB và mask polygon của một mẫu validation."""
    selected_row = select_gt_row(want_tumor)
    image_name = str(selected_row["image_id"])

    image = Image.open(
        BTXRD_ROOT / "images" / image_name
    ).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))

    sample_index = next(
        index
        for index, sample in enumerate(segmentation_ds.samples)
        if str(sample["image_id"]) == image_name
    )
    _, mask_tensor, _ = segmentation_ds[sample_index]
    mask = mask_tensor[0].numpy() > 0.5

    return image_name, np.asarray(image), mask


### Cell 3.1b — Tạo overlay polygon

**Mục đích:** tô vùng polygon bằng đỏ để dễ phát hiện annotation sai hoặc lệch.


In [ ]:
def build_gt_overlay(image_np, mask):
    """Tạo overlay đỏ cho polygon chẩn đoán."""
    overlay = image_np.copy()
    overlay[mask] = (
        0.45 * overlay[mask]
        + 0.55 * np.array([255, 30, 30])
    ).astype(np.uint8)
    return overlay


### Cell 3.1c — Hiển thị một trường hợp ground truth

**Mục đích:** ghép ảnh gốc, mask và overlay trong cùng một hàng.


In [ ]:
def show_gt_case(want_tumor):
    """Hiển thị ảnh, polygon và overlay chẩn đoán."""
    image_name, image_np, mask = load_gt_case(want_tumor)
    overlay = build_gt_overlay(image_np, mask)

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    panels = [
        (image_np, f"{image_name}\nẢnh gốc", None),
        (mask, "Polygon ground truth", "gray"),
        (overlay, "Overlay chỉ dùng chẩn đoán", None),
    ]

    for axis, (panel, title, cmap) in zip(axes, panels):
        axis.imshow(panel, cmap=cmap)
        axis.set_title(title)
        axis.axis("off")

    plt.tight_layout()
    plt.show()


### Cell 3.2 — Xem một ảnh có khối u

**Mục đích:** kiểm tra polygon của một mẫu `tumor_type > 0`.


In [ ]:
show_gt_case(want_tumor=True)


### Cell 3.3 — Xem một ảnh normal

**Mục đích:** kiểm tra rằng mẫu normal không chứa vùng polygon bất thường ngoài ý muốn.


In [ ]:
show_gt_case(want_tumor=False)


# 4. Chuẩn bị SAM và huấn luyện classifier

Classifier mức ảnh là nguồn tạo CAM. Profile `btxrd_best` cố định cấu hình chính: `tumor_type`, ảnh 320 px, batch 4, 6 epoch, seed 42 và inverse-frequency cross-entropy.


### Cell 4.1 — Chuẩn bị thư mục checkpoint SAM

**Mục đích:** tạo thư mục cha trước khi kiểm tra hoặc tải checkpoint.


In [ ]:
SAM_CHECKPOINT.parent.mkdir(
    parents=True,
    exist_ok=True,
)

print("Đường dẫn checkpoint SAM:", SAM_CHECKPOINT)


### Cell 4.2 — Tải SAM ViT-B nếu chưa có

**Mục đích:** tải checkpoint chính thức `sam_vit_b_01ec64.pth` chỉ khi file chưa tồn tại.

Kaggle cần bật Internet hoặc checkpoint phải được attach sẵn dưới dạng dataset.


In [ ]:
if not SAM_CHECKPOINT.exists():
    import urllib.request

    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/segment_anything/"
        "sam_vit_b_01ec64.pth",
        str(SAM_CHECKPOINT),
    )
else:
    print("Checkpoint SAM đã tồn tại, không tải lại.")


### Cell 4.3 — Kiểm tra kích thước checkpoint SAM

**Mục đích:** xác nhận file tồn tại và không phải file rỗng/hỏng rõ ràng.


In [ ]:
if not SAM_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Không tìm thấy checkpoint SAM: {SAM_CHECKPOINT}"
    )

sam_size_gib = round(
    SAM_CHECKPOINT.stat().st_size / 2**30,
    3,
)

print("Checkpoint SAM:", SAM_CHECKPOINT)
print("Kích thước:", sam_size_gib, "GiB")


### Cell 4.4 — Khai báo checkpoint classifier và lệnh huấn luyện

**Mục đích:** chỉ chuẩn bị lệnh, chưa chạy.

Các snapshot CAM được lưu tại epoch 1, 3 và 6 để theo dõi CAM thay đổi trong quá trình học.


In [ ]:
CLASSIFIER_CHECKPOINT = (
    CLASSIFIER_OUTPUT / "best_classifier.pt"
)

classifier_cmd = [
    sys.executable,
    "train_classifier.py",
    "--dataset",
    "btxrd",
    "--pipeline-profile",
    "btxrd_best",
    "--ram-root",
    str(BTXRD_ROOT),
    "--num-workers",
    str(NUM_WORKERS),
    "--save-cam-epochs",
    "1,3,6",
    "--cam-preview-count",
    "4",
    "--output-dir",
    str(CLASSIFIER_OUTPUT),
]

print("Lệnh huấn luyện classifier đã được chuẩn bị.")
print("Checkpoint kỳ vọng:", CLASSIFIER_CHECKPOINT)


### Cell 4.5 — Chạy huấn luyện classifier

**Mục đích:** gọi trực tiếp `train_classifier.py`.

Đặt `RUN_TRAIN_CLASSIFIER=False` khi muốn dùng checkpoint đã có.


In [ ]:
if RUN_TRAIN_CLASSIFIER:
    run_streaming(classifier_cmd)
else:
    print(
        "Bỏ qua huấn luyện vì RUN_TRAIN_CLASSIFIER=False."
    )


### Cell 4.6 — Xác nhận checkpoint classifier

**Mục đích:** dừng sớm nếu không có checkpoint để tránh lỗi khó hiểu ở các cell CAM phía sau.


In [ ]:
if not CLASSIFIER_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Không tìm thấy checkpoint classifier: "
        f"{CLASSIFIER_CHECKPOINT}"
    )

print(
    "Checkpoint classifier đã sẵn sàng:",
    CLASSIFIER_CHECKPOINT,
)


### Cell 4.7 — Đọc log huấn luyện

**Mục đích:** tải `training_log.csv` và hiển thị số liệu từng epoch.

Nếu file chưa có, cell chỉ in cảnh báo thay vì làm notebook dừng.


In [ ]:
training_log = CLASSIFIER_OUTPUT / "training_log.csv"

if training_log.exists():
    train_df = pd.read_csv(training_log)
    display(train_df)
else:
    train_df = pd.DataFrame()
    print(
        "Chưa có training_log.csv. "
        "Hãy chạy cell huấn luyện classifier trước."
    )


### Cell 4.8 — Khai báo cấu hình các đường cong huấn luyện

**Mục đích:** định nghĩa cặp cột và tiêu đề cho loss, accuracy và macro-F1.

Tách cấu hình khỏi cell vẽ giúp dễ bổ sung metric mới mà không lặp code.


In [ ]:
training_plot_specs = [
    (["train_loss", "val_loss"], "Mất mát cross-entropy"),
    (["train_acc", "val_acc"], "Độ chính xác"),
    (["train_f1", "val_f1"], "Macro-F1"),
]


### Cell 4.8a — Vẽ ba đường cong huấn luyện

**Mục đích:** dùng một cấu hình chung để giảm code lặp và giữ cách trình bày nhất quán.


In [ ]:
if not train_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(17, 4))

    for axis, (columns, title) in zip(axes, training_plot_specs):
        train_df[columns].plot(ax=axis, marker="o", title=title)
        axis.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("Không có log để vẽ đường cong huấn luyện.")


### Cell 4.9 — Nạp metadata checkpoint

**Mục đích:** đọc checkpoint trên CPU để kiểm tra metadata mà không chiếm VRAM.


In [ ]:
classifier_state = torch.load(
    CLASSIFIER_CHECKPOINT,
    map_location="cpu",
)

print("Các khóa metadata trong checkpoint:")
print(sorted(classifier_state.keys()))


### Cell 4.10 — Kiểm tra checkpoint đúng profile

**Mục đích:** xác nhận checkpoint là bài toán single-label 10 lớp với normalization ImageNet.

Assertion bảo vệ notebook khỏi việc vô tình dùng checkpoint của pipeline cũ hoặc dataset khác.


In [ ]:
expected_checkpoint_meta = {
    "task": "single-label",
    "target_columns": ["tumor_type"],
    "num_classes": 10,
    "normalization": "imagenet",
}

for key, expected_value in expected_checkpoint_meta.items():
    actual_value = classifier_state.get(key)

    print(
        f"{key}: thực tế={actual_value}, "
        f"kỳ vọng={expected_value}"
    )

    assert actual_value == expected_value

print("Checkpoint phù hợp với profile btxrd_best.")


### Cell 4.11 — Gom snapshot CAM theo ảnh

**Mục đích:** tìm các file dạng `cam_epoch<epoch>_<image>.png` và nhóm chúng theo ảnh để so sánh giữa các epoch.


In [ ]:
from collections import defaultdict
import re

cam_dir = CLASSIFIER_OUTPUT / "cam_preview"
cam_by_sample = defaultdict(list)

if cam_dir.exists():
    for cam_path in sorted(
        cam_dir.glob("cam_epoch*.png")
    ):
        match = re.match(
            r"cam_epoch(\d+)_(.+)\.png",
            cam_path.name,
        )

        if match:
            epoch = int(match.group(1))
            sample_stem = match.group(2)

            cam_by_sample[sample_stem].append(
                (epoch, cam_path)
            )

print("Số ảnh có snapshot CAM:", len(cam_by_sample))


### Cell 4.12 — Khởi tạo lưới snapshot CAM

**Mục đích:** tính số hàng/cột cần thiết và tạo figure trước khi điền từng ảnh CAM.

Nếu không có snapshot, `cam_rows` được đặt thành danh sách rỗng để cell sau chạy an toàn.


In [ ]:
if cam_by_sample:
    cam_rows = sorted(cam_by_sample.items())
    max_columns = max(len(entries) for _, entries in cam_rows)

    fig, axes = plt.subplots(
        len(cam_rows),
        max_columns,
        figsize=(3.2 * max_columns, 3.2 * len(cam_rows)),
        squeeze=False,
    )
else:
    cam_rows = []
    print("Không tìm thấy snapshot CAM để hiển thị.")


### Cell 4.12a — Điền ảnh CAM vào lưới

**Mục đích:** mỗi hàng là một ảnh, mỗi cột là một epoch đã lưu.


In [ ]:
for row_index, (sample_stem, entries) in enumerate(cam_rows):
    sorted_entries = sorted(entries)

    for column_index, (epoch, cam_path) in enumerate(sorted_entries):
        axis = axes[row_index, column_index]
        axis.imshow(Image.open(cam_path))
        axis.set_title(f"{sample_stem}\nepoch {epoch}")
        axis.axis("off")

    for column_index in range(len(sorted_entries), max_columns):
        axes[row_index, column_index].axis("off")

if cam_rows:
    plt.tight_layout()
    plt.show()


# 5. Truy vết một ảnh trước khi đưa vào SAM

Phần này dừng tại bước sinh box và point. Mục tiêu là tách riêng lỗi classifier/CAM/morphology khỏi lỗi của SAM.

Hai giao thức được chạy song song:

- `predicted`: dùng lớp do classifier dự đoán, phản ánh hành vi end-to-end.
- `ground_truth`: dùng nhãn ảnh-level thật, chỉ để kiểm tra khả năng định vị.


### Cell 5.1 — Import module LayerCAM và morphology

**Mục đích:** sử dụng đúng implementation trong repository hiện tại.


In [ ]:
from models.layercam import LayerCAM
from pseudo.generate_layercam import generate_fused_cam
from pseudo.tumor_morphology import (
    build_class_conditioned_components,
)
from generate_pseudo_masks import load_classifier


### Cell 5.2 — Nạp classifier từ checkpoint

**Mục đích:** sử dụng hàm `load_classifier` của generator để việc nạp model trong notebook giống pipeline thật.

Model d?ng thi?t b? CUDA n?u kh? d?ng; khi kh?ng c? CUDA, t? ??ng fallback sang CPU.


In [ ]:
classifier, checkpoint_task, classifier_normalization = load_classifier(
    CLASSIFIER_CHECKPOINT,
    fallback_num_classes=10,
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    expected_target_columns=["tumor_type"],
    expected_task="single-label",
    expected_num_classes=10,
)

classifier.eval()
checkpoint_meta = {"task": checkpoint_task, "normalization": classifier_normalization}

print("Classifier đã được nạp ở chế độ evaluation.")
display(pd.Series(checkpoint_meta).to_frame("giá trị"))


### Cell 5.3 — Tạo dataset truy vết

**Mục đích:** lấy ảnh validation theo đúng preprocessing của classification pipeline.


In [ ]:
trace_ds = build_classification_dataset(
    "btxrd",
    root=BTXRD_ROOT,
    split="val",
    target_columns=["tumor_type"],
    image_size=IMAGE_SIZE,
)

print("Số mẫu trong trace dataset:", len(trace_ds))


### Cell 5.4 — Nạp ảnh debug ở dạng tensor và RGB

**Mục đích:** chuẩn bị đồng thời:

- `trace_tensor`: tensor đã preprocessing để đưa vào classifier.
- `trace_rgb`: ảnh RGB 320 px dùng cho morphology và trực quan hóa.


In [ ]:
trace_index = next(
    index
    for index, sample in enumerate(trace_ds.samples)
    if str(sample["image_id"]) == DEBUG_IMAGE_NAME
)

trace_tensor, trace_target, _ = trace_ds[trace_index]

trace_rgb = np.asarray(
    Image.open(
        BTXRD_ROOT / "images" / DEBUG_IMAGE_NAME
    )
    .convert("RGB")
    .resize((IMAGE_SIZE, IMAGE_SIZE))
)

print("Tên ảnh:", DEBUG_IMAGE_NAME)
print("Shape tensor:", tuple(trace_tensor.shape))
print("Nhãn image-level:", int(trace_target))
print("Shape ảnh RGB:", trace_rgb.shape)


### Cell 5.5 — Hàm chuẩn hóa bản đồ về `[0, 1]`

**Mục đích:** đưa CAM tương phản về cùng miền giá trị để dễ hiển thị và threshold.

Số `1e-8` tránh chia cho 0 khi bản đồ có giá trị không đổi.


In [ ]:
def normalize_map(values):
    """Chuẩn hóa mảng số thực về miền [0, 1]."""
    values = np.asarray(
        values,
        dtype=np.float32,
    )

    return (
        values - values.min()
    ) / (
        values.max() - values.min() + 1e-8
    )


### Cell 5.6 — Hàm chọn lớp CAM

**Mục đích:** tách riêng logic chọn lớp khỏi logic tạo CAM.

- Với `predicted`, lấy lớp có xác suất cao nhất.
- Với `ground_truth`, lấy nhãn image-level.
- N?u l?p ???c ch?n l? normal (`0`), trace tr? CAM/support r?ng gi?ng pipeline production; kh?ng t? chuy?n sang tumor class kh?c.


In [ ]:
def select_cam_class(probabilities, target, protocol):
    """Chọn lớp dùng để tạo CAM theo giao thức yêu cầu."""
    if protocol == "predicted":
        selected_class = int(
            probabilities.argmax()
        )
    elif protocol == "ground_truth":
        selected_class = int(target)
    else:
        raise ValueError(
            f"Giao thức không hợp lệ: {protocol}"
        )

    # Normal không có vùng tumor để định vị.
    # V?i class normal, pipeline production b? qua ?nh v? tr? pseudo-mask r?ng.
    if selected_class == 0:
        return None

    return selected_class


### Cell 5.7 — Khởi tạo LayerCAM engine

**Mục đích:** đóng gói cấu hình LayerCAM chuẩn của profile:

- T? ??ng d?ng CUDA n?u kh? d?ng, n?u kh?ng th? ch?y CPU.
- Trọng số tầng `0.2 / 0.3 / 0.5`.
- Chỉ giữ gradient dương.


In [ ]:
def build_cam_engine(classifier_model):
    """Khởi tạo LayerCAM theo đúng trọng số của profile."""
    return LayerCAM(
        classifier_model,
        device=next(classifier_model.parameters()).device,
        layer_weights=[0.2, 0.3, 0.5],
        gradient_mode="positive",
    )


### Cell 5.7a — Hàm tạo hai loại CAM

**Mục đích:** tạo fused LayerCAM và CAM tương phản lớp-vs-normal từ cùng một engine.


In [ ]:
def generate_trace_cams(classifier_model, image_batch, selected_class):
    """Tạo fused LayerCAM và CAM tương phản lớp-vs-normal."""
    class_weights = np.zeros(10, dtype=np.float32)
    class_weights[selected_class] = 1.0
    cam_engine = build_cam_engine(classifier_model)

    fused_cam, _, _ = generate_fused_cam(
        cam_engine,
        image_batch,
        class_weights=class_weights,
        confidence_threshold=0.5,
    )
    contrast_output = cam_engine.cam_for_class_contrast(
        image_batch,
        selected_class,
        reference_index=0,
    )
    contrast_cam = normalize_map(
        contrast_output.cam.detach().cpu().numpy()[0]
    )

    return fused_cam, contrast_cam


### Cell 5.8 — Hàm tạo support, component, box và point

**Mục đích:** chạy morphology tại ba ngưỡng percentile `85`, `90`, `95`.

Mỗi kết quả chứa:

- `likelihood`: bản đồ likelihood sau xử lý.
- `support_mask`: vùng support nhị phân.
- `components`: tối đa 3 component cùng box và point.


In [ ]:
def generate_component_traces(
    image_rgb,
    contrast_cam,
):
    """Tạo component và prompt ở nhiều ngưỡng percentile."""
    component_traces = {}

    for percentile in [85, 90, 95]:
        (
            likelihood,
            support_mask,
            component_list,
        ) = build_class_conditioned_components(
            image_rgb,
            [contrast_cam],
            [1.0],
            cam_percentile=percentile,
            min_component_area=100,
            max_components=3,
            points_per_component=5,
            bbox_padding_ratio=0.02,
            negative_points_per_component=4,
        )

        component_traces[percentile] = {
            "likelihood": likelihood,
            "support_mask": support_mask,
            "components": component_list,
        }

    return component_traces


### Cell 5.9 — Hàm nạp một mẫu truy vết

**Mục đích:** tìm ảnh theo `image_id` trong classification dataset và trả tensor cùng nhãn image-level.

Cell này chưa chạy classifier hoặc tạo CAM.


In [ ]:
def load_trace_sample(image_name):
    """Tìm và nạp một mẫu classification theo tên ảnh."""
    sample_index = next(
        index
        for index, sample in enumerate(trace_ds.samples)
        if str(sample["image_id"]) == str(image_name)
    )
    return trace_ds[sample_index]


### Cell 5.9a — Hàm chạy classifier và lấy xác suất

**Mục đích:** tách inference classification khỏi các bước CAM/morphology.


In [ ]:
def predict_trace_probabilities(image_tensor):
    """Chạy classifier và trả vector xác suất."""
    device = next(classifier.parameters()).device
    image_batch = image_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        logits = classifier(image_batch)
        probabilities = torch.softmax(logits, dim=1)[0].cpu().numpy()

    return image_batch, probabilities


### Cell 5.9b — Hàm truy vết hoàn chỉnh trước SAM

**Mục đích:** ghép các helper nhỏ thành pipeline từ classification đến prompt.


In [ ]:
def trace_pre_sam(image_name=DEBUG_IMAGE_NAME, protocol="predicted"):
    """Truy vết một ảnh từ classifier đến prompt trước SAM."""
    image_tensor, target, _ = load_trace_sample(image_name)
    image_rgb = np.asarray(
        Image.open(BTXRD_ROOT / "images" / str(image_name)).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))
    )
    image_batch, probabilities = predict_trace_probabilities(image_tensor)

    selected_class = select_cam_class(probabilities, target, protocol)
    if selected_class is None:
        empty_cam = np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=np.float32)
        empty_components = {
            percentile: {"likelihood": empty_cam.copy(), "support_mask": np.zeros_like(empty_cam, dtype=np.uint8), "components": []}
            for percentile in [85, 90, 95]
        }
        return {
            "image": image_rgb, "target": int(target), "selected": None,
            "probs": probabilities, "cam": empty_cam, "contrast": empty_cam.copy(),
            "components": empty_components,
        }
    fused_cam, contrast_cam = generate_trace_cams(
        classifier,
        image_batch,
        selected_class,
    )
    component_traces = generate_component_traces(image_rgb, contrast_cam)

    return {
        "image": image_rgb,
        "target": int(target),
        "selected": selected_class,
        "probs": probabilities,
        "cam": fused_cam,
        "contrast": contrast_cam,
        "components": component_traces,
    }


### Cell 5.10 — Chạy truy vết với giao thức `predicted`

**Mục đích:** mô phỏng đúng hành vi triển khai end-to-end, trong đó classifier quyết định lớp tạo CAM.


In [ ]:
trace_predicted = trace_pre_sam(
    protocol="predicted"
)

print(
    "Lớp classifier chọn để tạo CAM:",
    trace_predicted["selected"],
)


### Cell 5.11 — Chạy truy vết với giao thức `ground_truth`

**Mục đích:** tách lỗi phân loại khỏi lỗi định vị bằng cách cấp nhãn image-level thật cho bước CAM.

Kết quả này chỉ là chẩn đoán, không phải kết quả triển khai.


In [ ]:
trace_ground_truth = trace_pre_sam(
    protocol="ground_truth"
)

print(
    "Lớp ground-truth dùng để tạo CAM:",
    trace_ground_truth["selected"],
)


### Cell 5.12 — In tóm tắt lớp và xác suất

**Mục đích:** so sánh nhãn thật với dự đoán classifier và xem xác suất của cả 10 lớp.


In [ ]:
print("Ảnh:", DEBUG_IMAGE_NAME)
print(
    "Nhãn image-level:",
    trace_predicted["target"],
)
print(
    "Lớp dự đoán dùng cho CAM:",
    trace_predicted["selected"],
)
print(
    "Lớp ground-truth dùng cho CAM:",
    trace_ground_truth["selected"],
)

probability_table = pd.DataFrame(
    {
        "class_index": range(
            len(trace_predicted["probs"])
        ),
        "class_name": TUMOR_TYPE_CLASS_NAMES,
        "probability": trace_predicted["probs"],
    }
).sort_values(
    "probability",
    ascending=False,
)

display(probability_table)


### Cell 5.13 — Hàm vẽ bốn panel CAM chính

**Mục đích:** hiển thị ảnh gốc, LayerCAM hợp nhất, CAM lớp-vs-normal và support percentile 85.

Box và point được tách sang helper riêng ở cell tiếp theo.


In [ ]:
def draw_cam_panels(axes, trace, title):
    """Vẽ ảnh gốc, fused CAM, contrast CAM và support p85."""
    axes[0].imshow(trace["image"])
    axes[0].set_title(f"Ảnh gốc\n{title}")

    axes[1].imshow(trace["image"])
    axes[1].imshow(trace["cam"], cmap="magma", alpha=0.48)
    axes[1].set_title("LayerCAM hợp nhất")

    axes[2].imshow(trace["image"])
    axes[2].imshow(trace["contrast"], cmap="jet", alpha=0.48)
    axes[2].set_title("CAM lớp-vs-normal")

    axes[3].imshow(trace["image"])
    axes[3].imshow(
        trace["components"][85]["support_mask"],
        cmap="Greens",
        alpha=0.45,
    )
    axes[3].set_title("Support percentile 85")


### Cell 5.13a — Hàm vẽ box và point

**Mục đích:** cô lập phần prompt visualization vì đây là phần dễ thay đổi khi thử chiến lược sinh điểm mới.


In [ ]:
def draw_prompt_panel(axis, trace):
    """Vẽ bounding box, positive point và negative point."""
    axis.imshow(trace["image"])

    for component in trace["components"][85]["components"]:
        x0, y0, x1, y1 = component["bbox"]
        axis.add_patch(
            plt.Rectangle(
                (x0, y0),
                x1 - x0,
                y1 - y0,
                fill=False,
                color="yellow",
                linewidth=2,
            )
        )

        positive = np.asarray(component["positive_points"])
        negative = np.asarray(component["negative_points"])

        if len(positive):
            axis.scatter(positive[:, 0], positive[:, 1], c="red", s=25)
        if len(negative):
            axis.scatter(negative[:, 0], negative[:, 1], c="cyan", s=20)

    axis.set_title("Box và các point")


### Cell 5.13b — Hàm ghép panel truy vết

**Mục đích:** tạo bố cục năm hình và gọi hai helper phía trên.


In [ ]:
def plot_pre_sam_trace(trace, title):
    """Hiển thị toàn bộ artifact trước khi gọi SAM."""
    fig, axes = plt.subplots(1, 5, figsize=(21, 4))

    draw_cam_panels(axes, trace, title)
    draw_prompt_panel(axes[4], trace)

    for axis in axes:
        axis.axis("off")

    plt.tight_layout()
    plt.show()


### Cell 5.14 — Hiển thị truy vết `predicted`

**Mục đích:** xem lỗi end-to-end thực tế bắt đầu từ classification hay từ localization.


In [ ]:
plot_pre_sam_trace(
    trace_predicted,
    "Lớp do classifier dự đoán",
)


### Cell 5.15 — Hiển thị truy vết `ground_truth`

**Mục đích:** quan sát chất lượng CAM khi loại bỏ lỗi chọn sai lớp.

Không dùng hình này như kết quả triển khai.


In [ ]:
plot_pre_sam_trace(
    trace_ground_truth,
    "Lớp ground-truth — chỉ chẩn đoán",
)


### Cell 5.16 — So sánh support ở percentile 85/90/95

**Mục đích:** đánh giá độ nhạy của vùng support đối với ngưỡng CAM.

- Percentile thấp thường tăng recall nhưng dễ leakage.
- Percentile cao thường giảm nhiễu nhưng có thể bỏ sót vùng tổn thương.


In [ ]:
fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 4),
)

for axis, percentile in zip(
    axes,
    [85, 90, 95],
):
    component_data = (
        trace_predicted["components"][percentile]
    )

    axis.imshow(trace_predicted["image"])
    axis.imshow(
        component_data["support_mask"],
        cmap="Greens",
        alpha=0.45,
    )

    component_count = len(
        component_data["components"]
    )

    axis.set_title(
        f"Predicted support p{percentile}\n"
        f"Số component={component_count}"
    )
    axis.axis("off")

plt.tight_layout()
plt.show()


# 6. Truy vết candidate SAM bằng generator thật

Các cell dưới đây gọi `generate_pseudo_masks.py --debug`. Vì vậy artifact hiển thị đến từ chính pipeline sản xuất, không phải một bản reimplementation đơn giản hóa trong notebook.


### Cell 6.1 — Tạo thư mục debug

**Mục đích:** đảm bảo generator có nơi ghi artifact của một ảnh.


In [ ]:
DEBUG_OUTPUT.mkdir(
    parents=True,
    exist_ok=True,
)

print("Thư mục debug:", DEBUG_OUTPUT)


### Cell 6.2 — Chuẩn bị lệnh debug một ảnh

**Mục đích:** chạy đúng ảnh đã chọn với các tùy chọn:

- `--debug`: lưu artifact trung gian.
- `--evaluate-prompt-quality`: ghi chỉ số chất lượng prompt và candidate.
- `--save-visuals-limit 1`: chỉ lưu visualization cho một ảnh.


In [ ]:
debug_cmd = [
    sys.executable,
    "generate_pseudo_masks.py",
    "--dataset",
    "btxrd",
    "--pipeline-profile",
    "btxrd_best",
    "--ram-root",
    str(BTXRD_ROOT),
    "--split",
    "val",
    "--classifier-checkpoint",
    str(CLASSIFIER_CHECKPOINT),
    "--sam-checkpoint",
    str(SAM_CHECKPOINT),
    "--image-list",
    str(DEBUG_IMAGE_LIST),
    "--max-images",
    "1",
    "--debug",
    "--evaluate-prompt-quality",
    "--save-visuals-limit",
    "1",
    "--output-dir",
    str(DEBUG_OUTPUT),
]

print("Lệnh debug một ảnh đã được chuẩn bị.")


### Cell 6.3 — Chạy generator debug

**Mục đích:** sinh CAM, support, prompt, toàn bộ candidate SAM, score và pseudo-mask cuối cho một ảnh.


In [ ]:
if RUN_SINGLE_IMAGE_DEBUG:
    run_streaming(debug_cmd)
else:
    print(
        "Bỏ qua chạy debug vì "
        "RUN_SINGLE_IMAGE_DEBUG=False."
    )


### Cell 6.4 — Xác định thư mục artifact của ảnh

**Mục đích:** tạo đường dẫn tới thư mục `debug/<image_stem>/` mà generator đã ghi.


In [ ]:
debug_case_dir = (
    DEBUG_OUTPUT
    / "debug"
    / Path(DEBUG_IMAGE_NAME).stem
)

print("Thư mục artifact của ảnh:", debug_case_dir)


### Cell 6.5 — Hàm hiển thị một artifact ảnh

**Mục đích:** chuẩn hóa cách kiểm tra file tồn tại và hiển thị ảnh với Matplotlib.


In [ ]:
def display_artifact(
    path,
    title=None,
    cmap=None,
):
    """Hiển thị artifact nếu file tồn tại."""
    if not path.exists():
        print("Thiếu artifact:", path)
        return

    artifact_image = Image.open(path)

    plt.figure(figsize=(4, 4))
    plt.imshow(artifact_image, cmap=cmap)
    plt.title(title or path.name)
    plt.axis("off")
    plt.show()


### Cell 6.6 — Hiển thị likelihood, seed và support

**Mục đích:** xem morphology biến CAM thành vùng support như thế nào trước khi tạo prompt SAM.

Các file `simple_*` giúp so sánh phiên bản cơ bản với phiên bản xử lý đầy đủ.


In [ ]:
debug_map_files = [
    "simple_tumor_likelihood.png",
    "simple_tumor_support.png",
]

for file_name in debug_map_files:
    display_artifact(
        debug_case_dir / file_name
    )


### Cell 6.7 — Hiển thị pseudo-mask cuối

**Mục đích:** xem mask sau khi chọn candidate và hậu xử lý morphology.


In [ ]:
final_mask_path = (
    DEBUG_OUTPUT
    / "masks"
    / f"{Path(DEBUG_IMAGE_NAME).stem}.png"
)

display_artifact(
    final_mask_path,
    title="Pseudo-mask cuối cùng",
    cmap="gray",
)


### Cell 6.8 — Đọc bảng score của candidate

**Mục đích:** tải `scores.json`, chuyển thành DataFrame và sắp xếp để dễ phân tích.

Mỗi hàng đại diện cho một candidate SAM trước khi chọn kết quả cuối.


In [ ]:
score_path = debug_case_dir / "scores.json"

if score_path.exists():
    score_records = json.loads(
        score_path.read_text(encoding="utf-8")
    )

    candidate_table = (
        pd.DataFrame.from_dict(
            score_records,
            orient="index",
        )
        .rename_axis("candidate")
        .reset_index()
    )

    display(candidate_table.head(12))
else:
    candidate_table = pd.DataFrame()

    print(
        "Chưa có scores.json. "
        "Hãy chạy cell generator debug trước."
    )


### Cell 6.9 — Tìm overlay của các candidate

**Mục đích:** lấy tối đa 12 ảnh overlay đầu tiên để đối chiếu trực quan với bảng score.


In [ ]:
overlay_paths = []

if debug_case_dir.exists():
    overlay_paths = sorted(
        debug_case_dir.glob("overlay_mask_*.png")
    )[:12]

print("Số overlay candidate tìm thấy:", len(overlay_paths))


### Cell 6.10 — Hiển thị lưới candidate SAM

**Mục đích:** phát hiện ba loại lỗi thường gặp:

- SAM không tạo được candidate tốt.
- Candidate tốt có tồn tại nhưng hàm chọn score sai.
- Candidate tốt bị hậu xử lý làm xấu đi.


In [ ]:
if overlay_paths:
    fig, axes = plt.subplots(
        3,
        4,
        figsize=(16, 12),
    )
    axes = axes.ravel()

    for axis, overlay_path in zip(
        axes,
        overlay_paths,
    ):
        axis.imshow(
            Image.open(overlay_path)
        )
        axis.set_title(overlay_path.name)
        axis.axis("off")

    for axis in axes[len(overlay_paths):]:
        axis.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print(
        "Không có overlay candidate để hiển thị."
    )


# 7. Giao thức end-to-end `predicted`

Đây là giao thức có thể triển khai: classifier tự dự đoán lớp và lớp đó quyết định CAM. Kết quả phải được báo cáo riêng, không trộn với protocol oracle.


### Cell 7.1 — Chuẩn bị lệnh sinh pseudo-mask `predicted`

**Mục đích:** chạy toàn bộ validation bằng profile `btxrd_best` và lưu kết quả trong thư mục riêng.


In [ ]:
PREDICTED_CMD = [
    sys.executable,
    "generate_pseudo_masks.py",
    "--dataset",
    "btxrd",
    "--pipeline-profile",
    "btxrd_best",
    "--ram-root",
    str(BTXRD_ROOT),
    "--split",
    "val",
    "--classifier-checkpoint",
    str(CLASSIFIER_CHECKPOINT),
    "--sam-checkpoint",
    str(SAM_CHECKPOINT),
    "--process-all",
    "--evaluate-prompt-quality",
    "--save-visuals-limit",
    "10",
    "--output-dir",
    str(PREDICTED_OUTPUT),
]

print("Lệnh predicted protocol đã được chuẩn bị.")


### Cell 7.2 — Chạy toàn bộ validation `predicted`

**Mục đích:** tạo pseudo-mask end-to-end cho mọi ảnh validation.


In [ ]:
if RUN_FULL_PREDICTED:
    run_streaming(PREDICTED_CMD)
else:
    print(
        "Bỏ qua predicted protocol vì "
        "RUN_FULL_PREDICTED=False."
    )


### Cell 7.3 — Chuẩn bị lệnh đánh giá `predicted`

**Mục đích:** so sánh pseudo-mask với polygon **sau khi generation đã hoàn tất**.

Polygon chỉ xuất hiện trong script đánh giá, không tham gia sinh mask.


In [ ]:
PREDICTED_EVAL = (
    EVAL_OUTPUT / "predicted.csv"
)

predicted_eval_cmd = [
    sys.executable,
    "evaluate_ramh1200_masks.py",
    "--dataset",
    "btxrd",
    "--ram-root",
    str(BTXRD_ROOT),
    "--split",
    "val",
    "--image-size",
    str(IMAGE_SIZE),
    "--pred-mask-root",
    str(PREDICTED_OUTPUT / "masks"),
    "--output-csv",
    str(PREDICTED_EVAL),
]

print("File đánh giá predicted:", PREDICTED_EVAL)


### Cell 7.4 — Chạy đánh giá `predicted`

**Mục đích:** tạo CSV metric cho giao thức triển khai.


In [ ]:
if RUN_FULL_PREDICTED:
    run_streaming(predicted_eval_cmd)
else:
    print(
        "Chưa chạy đánh giá predicted vì "
        "RUN_FULL_PREDICTED=False."
    )
    print("Lệnh đã chuẩn bị:", predicted_eval_cmd)


# 8. Giao thức định vị `ground_truth`

Giao thức này cấp nhãn `tumor_type` thật ở mức ảnh cho bước CAM để trả lời câu hỏi: *Nếu classifier chọn đúng lớp, localization tốt đến đâu?*

Đây không phải kết quả inference end-to-end.


### Cell 8.1 — Chuẩn bị lệnh `ground_truth`

**Mục đích:** dùng `--cam-target-class ground_truth` và ghi kết quả sang thư mục riêng.


In [ ]:
GROUND_TRUTH_CMD = [
    sys.executable,
    "generate_pseudo_masks.py",
    "--dataset",
    "btxrd",
    "--pipeline-profile",
    "btxrd_best",
    "--ram-root",
    str(BTXRD_ROOT),
    "--split",
    "val",
    "--classifier-checkpoint",
    str(CLASSIFIER_CHECKPOINT),
    "--sam-checkpoint",
    str(SAM_CHECKPOINT),
    "--process-all",
    "--cam-target-class",
    "ground_truth",
    "--evaluate-prompt-quality",
    "--save-visuals-limit",
    "10",
    "--output-dir",
    str(GROUND_TRUTH_OUTPUT),
]

print("Lệnh ground_truth protocol đã được chuẩn bị.")


### Cell 8.2 — Chạy toàn bộ validation `ground_truth`

**Mục đích:** tạo kết quả định vị oracle ở mức lớp ảnh.


In [ ]:
if RUN_FULL_GROUND_TRUTH:
    run_streaming(GROUND_TRUTH_CMD)
else:
    print(
        "Bỏ qua ground_truth protocol vì "
        "RUN_FULL_GROUND_TRUTH=False."
    )


### Cell 8.3 — Chuẩn bị lệnh đánh giá `ground_truth`

**Mục đích:** tính metric cho protocol chẩn đoán và lưu riêng khỏi `predicted.csv`.


In [ ]:
GROUND_TRUTH_EVAL = (
    EVAL_OUTPUT / "ground_truth.csv"
)

ground_truth_eval_cmd = [
    sys.executable,
    "evaluate_ramh1200_masks.py",
    "--dataset",
    "btxrd",
    "--ram-root",
    str(BTXRD_ROOT),
    "--split",
    "val",
    "--image-size",
    str(IMAGE_SIZE),
    "--pred-mask-root",
    str(GROUND_TRUTH_OUTPUT / "masks"),
    "--output-csv",
    str(GROUND_TRUTH_EVAL),
]

print(
    "File đánh giá ground_truth:",
    GROUND_TRUTH_EVAL,
)


### Cell 8.4 — Chạy đánh giá `ground_truth`

**Mục đích:** tạo CSV metric cho protocol chẩn đoán localization.


In [ ]:
if RUN_FULL_GROUND_TRUTH:
    run_streaming(ground_truth_eval_cmd)
else:
    print(
        "Chưa chạy đánh giá ground_truth vì "
        "RUN_FULL_GROUND_TRUTH=False."
    )
    print("Lệnh đã chuẩn bị:", ground_truth_eval_cmd)


# 9. Tổng hợp metric và phân rã lỗi

Phần này giữ metric `predicted` và `ground_truth` tách biệt, sau đó sử dụng `prompt_quality.csv` để xác định lỗi nằm ở CAM/support, candidate SAM, bước chọn candidate hay hậu xử lý.


### Cell 9.1 — Hàm đọc file tóm tắt đánh giá

**Mục đích:** chuyển CSV dạng hàng của script đánh giá thành dictionary metric.

Hàm trả dictionary rỗng nếu file chưa tồn tại để các cell tổng hợp vẫn chạy an toàn.


In [ ]:
def read_eval_summary(path):
    """Đọc file metric tổng hợp thành dictionary."""
    if not path.exists():
        return {}

    rows = pd.read_csv(
        path,
        header=None,
    )
    result = {}

    for row in rows.itertuples(
        index=False,
        name=None,
    ):
        if len(row) >= 5:
            primary_value = row[4]

            if primary_value == primary_value:
                result[str(row[0])] = primary_value
            elif len(row) > 5:
                result[str(row[0])] = row[5]
            else:
                result[str(row[0])] = np.nan

    return result


### Cell 9.2 — So sánh tóm tắt hai giao thức

**Mục đích:** đặt kết quả end-to-end và localization oracle cạnh nhau nhưng vẫn giữ nhãn protocol rõ ràng.

Khoảng cách lớn giữa hai hàng thường cho thấy lỗi classifier ảnh hưởng mạnh đến CAM.


In [ ]:
protocol_summary = pd.DataFrame(
    [
        {
            "protocol": "predicted",
            **read_eval_summary(PREDICTED_EVAL),
        },
        {
            "protocol": "ground_truth",
            **read_eval_summary(GROUND_TRUTH_EVAL),
        },
    ]
)

display(protocol_summary.T)


### Cell 9.3 — Hàm đọc `prompt_quality.csv`

**Mục đích:** nạp bảng phân rã lỗi của một output directory và thêm cột protocol.


In [ ]:
def load_quality(output_dir, protocol):
    """Đọc prompt_quality.csv và gắn tên protocol."""
    quality_path = (
        output_dir / "prompt_quality.csv"
    )

    if not quality_path.exists():
        return pd.DataFrame()

    quality_frame = pd.read_csv(
        quality_path
    )
    quality_frame.insert(
        0,
        "protocol",
        protocol,
    )

    return quality_frame


### Cell 9.4 — Ghép bảng chất lượng của hai protocol

**Mục đích:** tạo một DataFrame duy nhất phục vụ thống kê nhưng vẫn có cột `protocol` để không trộn ý nghĩa.


In [ ]:
quality = pd.concat(
    [
        load_quality(
            PREDICTED_OUTPUT,
            "predicted",
        ),
        load_quality(
            GROUND_TRUTH_OUTPUT,
            "ground_truth",
        ),
    ],
    ignore_index=True,
)

print("Số hàng prompt-quality:", len(quality))


### Cell 9.5 — Chọn các cột chẩn đoán chính

**Ý nghĩa nhanh:**

- `foreground_recall`, `foreground_iou`: chất lượng CAM/support.
- `oracle_best_single_dice`: candidate tốt nhất mà SAM có thể tạo.
- `selected_dice`: candidate thực tế được hàm score chọn.
- `support_loss_dice`: mất mát do support không bao phủ đủ.
- `selection_loss_dice`: mất mát do chọn sai candidate.
- `postprocess_delta_dice`: thay đổi do hậu xử lý.


In [ ]:
diagnostic_columns = [
    "protocol",
    "tumor_type",
    "foreground_iou",
    "foreground_recall",
    "foreground_precision",
    "point_hit_rate",
    "box_recall",
    "box_precision",
    "oracle_best_single_dice",
    "oracle_best_single_dice_clipped",
    "selected_dice",
    "support_loss_dice",
    "selection_loss_dice",
    "final_dice",
    "postprocess_delta_dice",
]


### Cell 9.6 — Thống kê mô tả prompt-quality

**Mục đích:** xem trung bình, độ lệch chuẩn, min/max và phân vị của từng chỉ số.

Nếu chưa có file, cell in hướng dẫn thay vì báo lỗi.


In [ ]:
if not quality.empty:
    available_columns = [
        column
        for column in diagnostic_columns
        if column in quality.columns
    ]

    display(
        quality[available_columns]
        .describe(include="all")
        .T
    )
else:
    print(
        "Chưa có prompt_quality.csv. "
        "Hãy chạy hai protocol với "
        "--evaluate-prompt-quality."
    )


### Cell 9.7 — Trung bình theo protocol và lớp

**Mục đích:** phát hiện lớp tumor nào thường gặp lỗi support hoặc selection.


In [ ]:
group_metric_columns = [
    "selected_dice",
    "final_dice",
    "support_loss_dice",
    "selection_loss_dice",
]

if not quality.empty:
    existing_group_metrics = [
        column
        for column in group_metric_columns
        if column in quality.columns
    ]

    grouped_quality = (
        quality.groupby(
            ["protocol", "tumor_type"]
        )[existing_group_metrics]
        .mean()
    )

    display(grouped_quality)
else:
    print("Không có dữ liệu để nhóm theo lớp.")


### Cell 9.8 — Khai báo metric dùng cho boxplot

**Mục đích:** xác định các cột bắt buộc và tiêu đề tương ứng trước khi vẽ.

Việc kiểm tra tập cột giúp tránh lỗi khi `prompt_quality.csv` được tạo bởi phiên bản code cũ.


In [ ]:
required_plot_columns = {
    "foreground_iou",
    "selection_loss_dice",
    "postprocess_delta_dice",
    "protocol",
}

boxplot_specs = [
    ("foreground_iou", "IoU foreground của CAM/prompt"),
    ("selection_loss_dice", "Mất mát do chọn candidate"),
    ("postprocess_delta_dice", "Thay đổi do hậu xử lý"),
]


### Cell 9.8a — Vẽ boxplot theo cấu hình

**Mục đích:** dùng vòng lặp thay cho ba đoạn lệnh lặp lại, giúp cell ngắn và dễ thêm metric mới.


In [ ]:
if not quality.empty and required_plot_columns.issubset(quality.columns):
    fig, axes = plt.subplots(1, 3, figsize=(17, 4))

    for axis, (column, title) in zip(axes, boxplot_specs):
        quality.boxplot(column=column, by="protocol", ax=axis)
        axis.set_title(title)
        axis.set_xlabel("")
        axis.grid(alpha=0.25)

    plt.suptitle("")
    plt.tight_layout()
    plt.show()
else:
    print("Thiếu dữ liệu hoặc cột cần thiết để vẽ boxplot.")


### Cell 9.9 — Tạo danh sách visualization theo thứ tự ưu tiên

**Mục đích:** chuẩn hóa vị trí có thể chứa fused LayerCAM, pseudo-mask hoặc overlay candidate.

Cell tiếp theo sẽ chọn đường dẫn đầu tiên thực sự tồn tại.


In [ ]:
def candidate_visual_paths(output_dir, image_name):
    """Tạo danh sách đường dẫn visualization theo thứ tự ưu tiên."""
    image_stem = Path(image_name).stem

    return [
        output_dir / "overlays" / f"{image_stem}_fused_layercam.png",
        output_dir / "masks" / f"{image_stem}.png",
        output_dir / "debug" / image_stem / "overlay_mask_0.png",
    ]


### Cell 9.9a — Chọn visualization đầu tiên tồn tại

**Mục đích:** tách logic tạo đường dẫn khỏi logic kiểm tra file để dễ sửa cấu trúc output về sau.


In [ ]:
def find_visual(output_dir, image_name):
    """Trả về visualization đầu tiên tồn tại, hoặc None."""
    return next(
        (
            path
            for path in candidate_visual_paths(output_dir, image_name)
            if path.exists()
        ),
        None,
    )


### Cell 9.10 — Nạp visualization của hai protocol

**Mục đích:** chuẩn bị ảnh gốc và artifact đại diện để so sánh định tính.


In [ ]:
predicted_visual = find_visual(
    PREDICTED_OUTPUT,
    DEBUG_IMAGE_NAME,
)
ground_truth_visual = find_visual(
    GROUND_TRUTH_OUTPUT,
    DEBUG_IMAGE_NAME,
)

comparison_image = (
    Image.open(
        BTXRD_ROOT
        / "images"
        / DEBUG_IMAGE_NAME
    )
    .convert("RGB")
    .resize((IMAGE_SIZE, IMAGE_SIZE))
)

print("Predicted visual:", predicted_visual)
print("Ground-truth visual:", ground_truth_visual)


### Cell 9.11 — Helper hiển thị ảnh hoặc thông báo thiếu file

**Mục đích:** tránh lặp logic kiểm tra đường dẫn ở hai panel `predicted` và `ground_truth`.


In [ ]:
def show_image_or_message(axis, image_path, title, missing_message):
    """Hiển thị ảnh nếu tồn tại; nếu không thì hiển thị thông báo."""
    if image_path:
        axis.imshow(Image.open(image_path))
    else:
        axis.text(0.5, 0.5, missing_message, ha="center")

    axis.set_title(title)
    axis.axis("off")


### Cell 9.11a — Khởi tạo panel so sánh

**Mục đích:** tạo bốn trục cho ảnh gốc, CAM và output của hai protocol.


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 4))

axes[0].imshow(comparison_image)
axes[0].set_title("Ảnh gốc")
axes[0].axis("off")

axes[1].imshow(comparison_image)
axes[1].imshow(
    trace_predicted["contrast"],
    cmap="jet",
    alpha=0.45,
)
axes[1].set_title("CAM của lớp dự đoán")
axes[1].axis("off")


### Cell 9.11b — Thêm output của hai protocol

**Mục đích:** hiển thị artifact nếu có; nếu thiếu thì ghi rõ trạng thái ngay trên panel.


In [ ]:
show_image_or_message(
    axes[2],
    predicted_visual,
    "Đầu ra giao thức predicted",
    "Thiếu predicted visual",
)

show_image_or_message(
    axes[3],
    ground_truth_visual,
    "Đầu ra GT-class — chẩn đoán",
    "Thiếu ground_truth visual",
)

plt.tight_layout()
plt.show()


# 10. Nhánh fully supervised bằng ground-truth — chạy đối chứng

Phần này **không thay thế và không sửa pipeline pseudo-mask ở các phần trước**. Đây là một nhánh độc lập:

```text
Train:     Image(train) + polygon GT(train) → U-Net fully supervised
Inference: Image(val) → U-Net → predicted mask
Evaluate:  predicted mask ↔ polygon GT(val)
```

Source `BTXRDSegmentationDataset` rasterize polygon LabelMe thành binary tumor mask. Nhánh WSSS vẫn chạy và lưu artifact trong các thư mục cũ; output GT-U-Net được tách riêng.

### Cell 10.1 — Khai báo cấu hình fully supervised

Cấu hình được ghi tường minh để lần chạy Kaggle có thể tái lập. Việc chọn model vẫn dựa trên validation; test chưa được mở trong notebook debug.

In [ ]:
GROUND_TRUTH_UNET_CMD = [
    sys.executable, "train_segmentation.py",
    "--dataset", "btxrd",
    "--ram-root", str(BTXRD_ROOT),
    "--train-split", "train",
    "--val-split", "val",
    "--image-size", str(IMAGE_SIZE),
    "--batch-size", "4",
    "--epochs", "50",
    "--lr", "1e-4",
    "--weight-decay", "1e-4",
    "--seed", "42",
    "--num-workers", str(NUM_WORKERS),
    "--early-stop-patience", "10",
    "--output-dir", str(GROUND_TRUTH_UNET_OUTPUT),
]

print("Output GT-U-Net:", GROUND_TRUTH_UNET_OUTPUT)
print("Lệnh:", " ".join(map(str, GROUND_TRUTH_UNET_CMD)))

### Cell 10.2 — Huấn luyện GT-U-Net

In [ ]:
if RUN_TRAIN_GROUND_TRUTH_UNET:
    run_streaming(GROUND_TRUTH_UNET_CMD)
else:
    print("RUN_TRAIN_GROUND_TRUTH_UNET=False; pipeline pseudo-mask vẫn giữ nguyên.")

### Cell 10.3 — Vẽ learning curve của nhánh GT

In [ ]:
gt_training_log = GROUND_TRUTH_UNET_OUTPUT / "training_log.csv"
if gt_training_log.exists():
    gt_history = pd.read_csv(gt_training_log)
    display(gt_history)
    best_row = gt_history.loc[gt_history["val_dice"].idxmax()]
    print("Best epoch:", int(best_row["epoch"]))
    print("Best val Dice:", float(best_row["val_dice"]))
    print("Val IoU at best Dice:", float(best_row["val_iou"]))
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    gt_history.plot(x="epoch", y=["train_loss", "val_loss"], ax=axes[0], title="GT-U-Net loss")
    gt_history.plot(x="epoch", y=["train_dice", "val_dice"], ax=axes[1], title="GT-U-Net Dice")
    gt_history.plot(x="epoch", y=["train_iou", "val_iou"], ax=axes[2], title="GT-U-Net IoU")
    for axis in axes: axis.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print("Chưa có training_log.csv của GT-U-Net.")

### Cell 10.4 — Đánh giá checkpoint GT-U-Net trên validation GT

Evaluator báo Dice/IoU riêng trên ảnh tumor và specificity trên ảnh normal, tránh pooled Dice bị nâng bởi các cặp mask rỗng.

In [ ]:
GROUND_TRUTH_UNET_EVAL_CSV = UNET_EVAL_OUTPUT / "ground_truth_unet_val.csv"
GROUND_TRUTH_UNET_EVAL_JSON = UNET_EVAL_OUTPUT / "ground_truth_unet_val.json"
GROUND_TRUTH_UNET_CHECKPOINT = GROUND_TRUTH_UNET_OUTPUT / "best_unet.pt"

ground_truth_unet_eval_cmd = [
    sys.executable, "evaluate_unet.py",
    "--dataset", "btxrd",
    "--ram-root", str(BTXRD_ROOT),
    "--split", "val",
    "--checkpoint", str(GROUND_TRUTH_UNET_CHECKPOINT),
    "--image-size", str(IMAGE_SIZE),
    "--batch-size", "4",
    "--num-workers", str(NUM_WORKERS),
    "--output-csv", str(GROUND_TRUTH_UNET_EVAL_CSV),
    "--output-json", str(GROUND_TRUTH_UNET_EVAL_JSON),
]

if RUN_EVALUATE_GROUND_TRUTH_UNET:
    if not GROUND_TRUTH_UNET_CHECKPOINT.exists():
        raise FileNotFoundError(GROUND_TRUTH_UNET_CHECKPOINT)
    run_streaming(ground_truth_unet_eval_cmd)
else:
    print("RUN_EVALUATE_GROUND_TRUTH_UNET=False")

### Cell 10.5 — Kết quả nhánh GT để ghép vào bảng so sánh

In [ ]:
if GROUND_TRUTH_UNET_EVAL_JSON.exists():
    gt_unet_summary = json.loads(GROUND_TRUTH_UNET_EVAL_JSON.read_text(encoding="utf-8"))
    gt_comparison_row = pd.DataFrame([{
        "method": "Fully supervised U-Net (polygon GT train)",
        "tumor_dice": gt_unet_summary["mean_tumor_dice"],
        "tumor_iou": gt_unet_summary["mean_tumor_iou"],
        "tumor_sensitivity": gt_unet_summary["tumor_detection_sensitivity"],
        "normal_specificity": gt_unet_summary["normal_specificity"],
        "normal_fpr": gt_unet_summary["normal_false_positive_rate"],
    }])
    display(gt_comparison_row)
else:
    print("Chưa có summary GT-U-Net để đưa vào bảng đối chứng.")

### Cell 10.6 — Bảng đối chứng cuối sau khi cả hai pipeline hoàn tất

Cell này lấy output `predicted` của pipeline hiện tại và output GT-U-Net vừa chạy, rồi đặt cạnh nhau bằng các metric có cùng định nghĩa. Protocol CAM `ground_truth` ở phần 8 chỉ là chẩn đoán oracle nên không đưa vào bảng kết quả chính.

In [ ]:
comparison_rows = []

# Kết quả end-to-end của pipeline pseudo-mask hiện tại
pseudo_summary = read_eval_summary(PREDICTED_EVAL)
if pseudo_summary:
    comparison_rows.append({
        "method": "Current pipeline (predicted pseudo-mask)",
        "training_supervision": "image-level labels",
        "tumor_dice": pseudo_summary.get("end_to_end_mean_tumor_dice", np.nan),
        "tumor_iou": pseudo_summary.get("end_to_end_mean_tumor_iou", np.nan),
        "normal_specificity": pseudo_summary.get("specificity_empty_mask_rate", np.nan),
        "normal_fpr": pseudo_summary.get("false_positive_rate", np.nan),
    })

# Kết quả của nhánh fully supervised chạy sau pipeline hiện tại
if GROUND_TRUTH_UNET_EVAL_JSON.exists():
    gt_summary = json.loads(GROUND_TRUTH_UNET_EVAL_JSON.read_text(encoding="utf-8"))
    comparison_rows.append({
        "method": "Fully supervised U-Net (polygon GT)",
        "training_supervision": "pixel-level polygon masks",
        "tumor_dice": gt_summary["mean_tumor_dice"],
        "tumor_iou": gt_summary["mean_tumor_iou"],
        "normal_specificity": gt_summary["normal_specificity"],
        "normal_fpr": gt_summary["normal_false_positive_rate"],
    })

final_comparison = pd.DataFrame(comparison_rows)
display(final_comparison)

if len(final_comparison) == 2:
    plot_columns = ["tumor_dice", "tumor_iou", "normal_specificity"]
    axis = final_comparison.set_index("method")[plot_columns].plot.bar(
        figsize=(11, 5), ylim=(0, 1), grid=True
    )
    axis.set_title("BTXRD validation: current pipeline vs fully supervised GT-U-Net")
    axis.set_ylabel("score")
    plt.xticks(rotation=10, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Cần chạy xong pipeline predicted và GT-U-Net để có đủ hai hàng đối chứng.")

# 11. Lưu manifest và kiểm tra tái lập

Manifest ghi lại các đường dẫn và cấu hình trọng yếu của lần chạy. File này hữu ích khi đối chiếu kết quả giữa Kaggle, Colab và máy local.


### Cell 11.1 — Tạo manifest

**Mục đích:** gom thông tin môi trường và artifact vào một dictionary.

Các khóa được giữ bằng tiếng Anh để thuận tiện cho script đọc tự động; phần mô tả giá trị dùng tiếng Việt.


In [ ]:
manifest = {
    "git_branch_requested": GIT_BRANCH,
    "dataset_root": str(BTXRD_ROOT),
    "classifier_checkpoint": str(
        CLASSIFIER_CHECKPOINT
    ),
    "sam_checkpoint": str(SAM_CHECKPOINT),
    "profile": "btxrd_best",
    "image_size": IMAGE_SIZE,
    "sam_image_size": SAM_IMAGE_SIZE,
    "predicted_output": str(
        PREDICTED_OUTPUT
    ),
    "ground_truth_unet_output": str(GROUND_TRUTH_UNET_OUTPUT),
    "unet_eval_output": str(UNET_EVAL_OUTPUT),
    "ground_truth_output": str(
        GROUND_TRUTH_OUTPUT
    ),
    "predicted_eval": str(
        PREDICTED_EVAL
    ),
    "ground_truth_eval": str(
        GROUND_TRUTH_EVAL
    ),
    "polygon_usage": (
        "Chỉ dùng cho đánh giá/chẩn đoán và nhánh fully supervised đối chứng"
    ),
    "test_tuning": False,
}

### Cell 11.2 — Ghi manifest ra JSON

**Mục đích:** lưu cấu hình tái lập vào `notebook_artifacts.json`.


In [ ]:
manifest_path = (
    OUTPUT_ROOT / "notebook_artifacts.json"
)

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("Đã lưu manifest tại:", manifest_path)


### Cell 11.3 — Hiển thị manifest và trạng thái cuối

**Mục đích:** kiểm tra nhanh mọi đường dẫn trước khi kết thúc phiên debug.


In [ ]:
display(
    pd.Series(
        manifest,
        name="giá trị",
    ).to_frame()
)

print(
    "Notebook hoàn tất khi các artifact được yêu cầu "
    "đã tồn tại trong:",
    OUTPUT_ROOT,
)


# Checklist diễn giải kết quả

- Chỉ báo cáo `predicted` như kết quả inference end-to-end.
- Chỉ dùng `ground_truth` như chẩn đoán localization/oracle ở mức lớp ảnh.
- Không gọi macro-F1 của classifier là metric định vị CAM.
- `foreground_recall` thấp hoặc `support_loss_dice` cao thường cho thấy lỗi CAM/morphology.
- Support tốt nhưng `oracle_best_single_dice` thấp thường cho thấy prompt hoặc SAM không tạo được candidate phù hợp.
- `oracle_best_single_dice` tốt nhưng `selection_loss_dice` cao cho thấy hàm chọn candidate cần cải thiện.
- `postprocess_delta_dice` có độ lớn cao cho thấy hậu xử lý đang thay đổi mask quá mạnh.
- Polygon chỉ được đọc trong các cell trực quan hóa, đánh giá và baseline oracle đã tắt mặc định.
- Dùng validation để debug; chỉ mở tập test một lần cho báo cáo cuối cùng sau khi khóa toàn bộ cấu hình.
